Install and import libraries

In [ ]:
# Data Manipulation
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import plotly.express as px

# Machine Learning (for forecasting later)
from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
# Display settings
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

Load dataset

In [ ]:
df = pd.read_excel("sample_-_superstore.xls")

In [ ]:
df.head()

Explore the dataset

In [ ]:
# Shape of the dataset
print("Rows and Columns:", df.shape)

# Columns names
print("\nColumns")
print(df.columns)

# Data types
print("\nData Types")
print(df.columns)

# Basic statistics
print("\nSummary Statistics:")
df.describe(include="all")

Check for missing values

In [ ]:
df.isnull().sum()

Check for Duplicate records

In [ ]:
# Number of duplicate rows
print("Duplicate Rows:", df.duplicated().sum())

Make a copy of the dataset

In [ ]:
sales = df.copy()

View dataset information

In [ ]:
sales.info()

Look for

* Number of rows and columns
* Data types
* Missing values

In [ ]:
print("Duplicates before: ", sales.duplicated().sum())

sales = sales.drop_duplicates()

print("Duplicates after: ", sales.duplicated().sum())

Handling missing values

In [ ]:
sales.isnull().sum()

If there are only a few missing values:

In [ ]:
sales = sales.dropna()

If a numeric column has missing values

In [ ]:
sales["Sales"] = sales["Sales"].fillna(sales["Sales"].mean())

If a categorical column has missing values

In [ ]:
sales["Category"] = sales["Sales"].fillna("Unknown")

Convert Dates

Business analysis depends heavily on dates.

In [ ]:
sales["Order Date"] = pd.to_datetime(sales["Order Date"])
sales["Ship Date"] = pd.to_datetime(sales["Ship Date"])

Verify:

In [ ]:
sales.dtypes

Create business features

These features make analysis much easier

Year

In [ ]:
sales["Year"] = sales["Order Date"].dt.year

Month Name

In [ ]:
sales["Month"] = sales["Order Date"].dt.month

Month Number

In [ ]:
sales["Month No"] = sales["Order Date"].dt.month

Quarter

In [ ]:
sales["Quarter"] = sales["Order Date"].dt.quarter

Day of week

In [ ]:
sales["Day"] = sales["Order Date"].dt.day_name()

Create Business KPIs

Profit Margin (%)

In [ ]:
sales["Profit Margin"] = (
    sales["Profit"] / sales["Sales"]
) * 100

Revenue per Item

In [ ]:
sales["Revenue per Item"] = (
    sales["Sales"] / sales["Quantity"]
)

Shopping Time

In [ ]:
sales["Shipping Days"] = (
    sales["Ship Date"] - sales["Order Date"]
).dt.days

Inspect the New Dataset

In [ ]:
sales.head()

Executive KPIs

In [ ]:
total_revenue = sales["Sales"].sum()

# Total Profit
total_profit = sales["Profit"].sum()

# Total Orders
total_orders = sales["Order ID"].nunique()

# Total Customers
total_customers = sales["Customer ID"].nunique()

# Average Order Value
# Calculate a single scalar for Average Order Value (Total Revenue / Total Orders)
average_order_value = total_revenue / total_orders

print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Profit: ${total_profit:,.2f}")
print(f"Total Orders: {total_orders}")
print(f"Total Customers: {total_customers}")
print(f"Average Order Value: ${average_order_value:,.2f}")

Monthly Revenue Trend

This helps identify seasonal patterns

In [ ]:
monthly_sales = (
   sales.groupby(["Year","Month No","Month"])["Sales"]
   .sum()
   .reset_index()
   .sort_values(["Year", "Month No"])
)

monthly_sales

Plot the trend

In [ ]:
plt.figure(figsize=(12,5))

plt.plot(
    monthly_sales["Month"],
    monthly_sales["Sales"],
    marker="o"
)

plt.title("Monthly Revenue")
plt.xlabel("Month")
plt.ylabel("Revenue")
plt.xticks(rotation=45)

plt.show()

Revenue by Region

In [ ]:
region_sales = (
    sales.groupby("Region")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

region_sales

Bar Chart

In [ ]:
region_sales.plot(
   kind="bar",
   figsize=(8,5)
)

plt.title("Revenue by Region")
plt.ylabel("Revenue")

plt.show()

Profit by Category

In [ ]:
category_profit = (
    sales.groupby("Category")["Profit"]
    .sum()
    .sort_values(ascending=False)
)

category_profit

Visualization

In [ ]:
category_profit.plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Profit by Category")
plt.ylabel("Profit")

plt.show()

Top 10 Customers

In [ ]:
top_customers = (
    sales.groupby("Customer Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_customers

Visualization

In [ ]:
top_customers.plot(
    kind="barh",
    figsize=(8,6)
)

plt.title("Top 10 Customers by Revenue")
plt.xlabel("Revenue")

plt.show()

Top 10 Products

In [ ]:
top_products = (
    sales.groupby("Product Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_products

Visualization

In [ ]:
top_products.plot(
    kind="barh",
    figsize=(10,6)
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue")

plt.show()

Profit Margin Analysis

* Let's see how predictable each product category is

In [ ]:
category_summary = (
    sales.groupby("Category")
    .agg(
        Revenue=("Sales","sum"),
        Profit=("Profit","sum")
    )
)

category_summary["Profit Margin (%)"] = (
    category_summary["Profit"] /
    category_summary["Revenue"]
) * 100

category_summary

Visualize profit margin

In [ ]:
category_summary["Profit Margin (%)"].plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Profit Margin by Category")
plt.ylabel("Profit Margin (%)")

plt.show()

Sales vs Profit

This helps determine whether higher sales actually lead to higher profits.

In [ ]:
plt.figure(figsize=(8,5))

plt.scatter(
    sales["Sales"],
    sales["Profit"]
)

plt.title("Sales vs Profit")
plt.xlabel("Sales")
plt.ylabel("Profit")

plt.show()

State-wise Sales Analysis

In [ ]:
state_sales = (
    sales.groupby("State")["Sales"]
    .sum()
    .sort_values(ascending=False)
)

state_sales.head(10)

Sub-Category Analysis

* This gives more detail than the broader product categories.

In [ ]:
subcategory = (
    sales.groupby("Sub-Category")
    .agg(
        Revenue=("Sales","sum"),
        Profit=("Profit","sum")
    )
    .sort_values("Revenue", ascending=False)
)

subcategory.head(10)

Visualization

In [ ]:
subcategory["Revenue"].head(10).plot(
    kind="barh",
    figsize=(10,6)
)

plt.title("Top 10 Sub-Categories")
plt.xlabel("Revenue")

plt.show()

Discount impact

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    sales["Discount"],
    sales["Profit"]
)

plt.title("Discount vs Profit")
plt.xlabel("Discount")
plt.ylabel("Profit")

plt.show()

This chart often reveals that:

* Higher discounts tend to reduce profits
* Some heavily discounted orders even result in losses.

Shipping performance

* Let's calculate the average shipping time.

In [ ]:
sales["Shipping Days"].describe()

Average shipping days:

In [ ]:
sales["Shipping Days"].mean()

Shipping time by shipping mode:

In [ ]:
sales.groupby("Ship Mode")["Shipping Days"].mean()

Pareto Analysis (80/20 Rule)

The Pareto principle states that roughly 80% of revenue often comes from 20% of customers. Let's test whether that's time for our dataset

Calculate Customer Revenue

In [ ]:
customer_sales = (
    sales.groupby("Customer Name")["Sales"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

customer_sales.columns = ["Customer Name", "Revenue"]

Calculate Cumulative Revenue

In [ ]:
customer_sales["Cumulative Revenue"] = customer_sales["Revenue"].cumsum()

customer_sales["Cumulative Percentage"] = (
    customer_sales["Cumulative Revenue"] /
    customer_sales["Revenue"].sum()
) * 100

customer_sales.head()

Plot the Pareto chart

In [ ]:
plt.figure(figsize=(12,6))

plt.bar(
    customer_sales["Customer Name"],
    customer_sales["Revenue"]
)

plt.plot(
    customer_sales["Cumulative Percentage"],
    color="red",
    marker="o"
)

plt.axhline(80, color="green", linestyle="--")

plt.xticks([])

plt.title("Pareto Analysis of Customer Revenue")
plt.ylabel("Revenue / Cumulative %")

plt.show()

Customer segmentation

Let's classify customers into High, Medium, and Low spenders.

In [ ]:
customer_summary = (
    sales.groupby("Customer Name")
    .agg(
        Revenue=("Sales","sum"),
        Orders=("Order ID","nunique"),
        Profit=("Profit","sum")
    )
)

customer_summary.head()

Create spending segments

In [ ]:
customer_summary["Segment"] = pd.qcut(
    customer_summary["Revenue"],
    q=3,
    labels=["Low Value","Medium Value","High Value"]
)

customer_summary.head()

Count customers per segment

In [ ]:
customer_summary["Segment"].value_counts()

Visualize the segments

In [ ]:
customer_summary["Segment"].value_counts().plot(
    kind="bar",
    figsize=(7,5)
)

plt.title("Customer Segments")
plt.ylabel("Number of Customers")

plt.show()

Interactive dashboard with poorly

Interactive charts are much more engaging them static ones

Monthly Revenue

In [ ]:
fig = px.line(
    monthly_sales,
    x="Month",
    y="Sales",
    title="Monthly Revenue Trend",
    markers=True
)

fig.show()

Revenue by Region

In [ ]:
fig = px.bar(
    region_sales.reset_index(),
    x="Region",
    y="Sales",
    title="Revenue by Region"
)

fig.show()

Category Revenue

In [ ]:
category_sales = (
    sales.groupby("Category")["Sales"]
    .sum()
    .reset_index()
)

fig = px.pie(
    category_sales,
    names="Category",
    values="Sales",
    title="Revenue by Category"
)

fig.show()

Prepare the Data for forecasting.

We'll forecast daily sales

In [ ]:
# Aggregate sales by order date
daily_sales = (
    sales.groupby("Order Date")["Sales"]
    .sum()
    .reset_index()
)

daily_sales.head()

Create a time index

Machine Learning models work with numbers rather than dates.

In [ ]:
daily_sales["Day"] = range(len(daily_sales))

daily_sales.head()

Train a Linear Regression model

In [ ]:
X = daily_sales[["Day"]]
y = daily_sales["Sales"]

model = LinearRegression()
model.fit(X,y)

print("Model trained successfully!")

Predict historical sales

In [ ]:
daily_sales["Predicted Sales"] = model.predict(X)

daily_sales.head()

Compare Actual vs Predicted Sales

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(
    daily_sales["Order Date"],
    daily_sales["Sales"],
    label="Actual Sales"
)

plt.plot(
    daily_sales["Order Date"],
    daily_sales["Predicted Sales"],
    label="Predicted Sales"
)

plt.title("Actual vs Predicted Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

Forecast the next 30 days

In [ ]:
feature_days = pd.DataFrame({
     "Day": range(
         len(daily_sales),
         len(daily_sales) + 30
     )
})

future_predictions = model.predict(feature_days)

future_predictions

Create Future Dates

In [ ]:
future_dates = pd.date_range(
    start = daily_sales["Order Date"].max() + pd.Timedelta(days=1),
    periods=30
)

forecast = pd.DataFrame({
    "Date": future_dates,
    "Forecast Sales": future_predictions
})

forecast.head()

Pot the Forecast

In [ ]:
plt.figure(figsize=(14,6))

plt.plot(
    daily_sales["Order Date"],
    daily_sales["Sales"],
    label="Historical Sales"
)

plt.plot(
    forecast["Date"],
    forecast["Forecast Sales"],
    label="Forecast",
    linestyle="--"
)

plt.title("30-Day Sales Forecast")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

Evaluate the model

Let's see how well the model fits the historical data.

In [ ]:
mae = mean_absolute_error(
    daily_sales["Sales"],
    daily_sales["Predicted Sales"]
)

mse = mean_squared_error(
    daily_sales["Sales"],
    daily_sales["Predicted Sales"]
)

r2 = r2_score(
   daily_sales["Sales"],
   daily_sales["Predicted Sales"]
)

print(f"Mean Absolute Error: {mae:.2f}")
print(f"Mean Squared Error: {mse}")
print(f"R^2 Score: {r2:.4f}")